In [ ]:
#%pip install svy-io
#https://svylab.com/learn/workshops/meps-2025/
#https://svylab.com/docs/svy/tutorials/estimation.html

In [ ]:
import os
os.environ.pop("POLARS_FEATURE_FLAGS", None)
os.environ["POLARS_SKIP_CPU_CHECK"] = "1"
import svy
import polars as pl
import pandas as pd
import numpy as np
from io import StringIO
import datetime as dt
import pyreadstat
import pyreadr


In [ ]:
#Load survey data
def read_dta(path):
    # Use pyreadstat to avoid native parser crashes in some kernels/CPUs.
    df, _meta = pyreadstat.read_dta(path)
    return pl.from_pandas(df)


In [ ]:
smp_data = read_dta("data/survey_data.dta")

In [ ]:
# Diagnostic: verify data type and columns before building survey sample
print("smp_data type:", type(smp_data))

if isinstance(smp_data, pl.DataFrame):
    print("Detected Polars DataFrame")
    print("Rows:", smp_data.height, "Cols:", smp_data.width)
    print("Schema:", smp_data.schema)
else:
    print("WARNING: smp_data is not a Polars DataFrame.")
    print("Expected: polars.DataFrame for svy.Sample")
    if hasattr(smp_data, "columns"):
        print("Columns:", list(smp_data.columns))

In [ ]:
smp_data = smp_data.with_columns(
    pl.col("psu").cast(pl.String).cast(pl.Categorical)
)


In [ ]:
#set negative income values to None (missing) to avoid skewing the mean calculation
smp_data = smp_data.with_columns(
    pl.when(pl.col("income") < 0)
      .then(None)
      .otherwise(pl.col("income"))
      .alias("income")
)


In [ ]:
# Define the survey design
smp_design = svy.Design(
    stratum=("strata"),  # Stratification variables
    psu="psu",                          # Primary sampling units
    wgt="wt_int"                          # Survey weights
)

# Create the sample object
svy_data = svy.Sample(data=smp_data, design=smp_design)

In [ ]:
# Complex conditions - remove records with negative income values
svy_data_filtered = svy_data.wrangling.filter_records([
    svy.col("income") > 0
])
print(svy_data_filtered.data)

In [ ]:
# Estimate population mean of income with design-based standard error
mean_income = svy_data_filtered.estimation.mean(y="income")  
print(mean_income)



In [ ]:
mean_income.to_polars()